# NLP Tutorial 1 — French Environmental Corpus (Group 3, French)

**Project:** *French Environmental Text Intelligence — automatic classification and information retrieval in French environmental texts.*

This week builds the corpus that the project will reuse later (tokenization, n-grams, TF-IDF, classification, search).

| | |
|---|---|
| Language | French (`fr_core_news_md`) |
| Text type | Introductions of French Wikipedia articles |
| Categories | émissions · énergies renouvelables · pollution · politique climatique |
| Licence | CC BY-SA 4.0 |

Run with **Restart and run all** before saving (section 6.7).

## 1. Setup — spaCy and the French model

In [ ]:
# Uncomment on Google Colab or on a new machine:
# !pip install -U spacy
# !python -m spacy download fr_core_news_md

In [ ]:
import spacy
from spacy import displacy
from pathlib import Path
from collections import Counter
import csv, re

print('spaCy version:', spacy.__version__)
nlp = spacy.load('fr_core_news_md')
print('Model:', nlp.meta['name'], nlp.meta['version'])
print('Pipeline:', nlp.pipe_names)

### Output for one sentence (hand-in item 1)

In [ ]:
sentence = "L'ADEME estime que les émissions de gaz à effet de serre de la France ont baissé de 5,8 % en 2023."
doc = nlp(sentence)
print(f"{'TOKEN':<14}{'LEMMA':<14}{'POS':<8}{'DEP':<12}HEAD")
for t in doc:
    print(f"{t.text:<14}{t.lemma_:<14}{t.pos_:<8}{t.dep_:<12}{t.head.text}")
print('\nEntities:', [(e.text, e.label_) for e in doc.ents])

### French-specific check: elision and contractions
Discussion question 2 asks how each language handles contractions such as *l'homme*. Let's see how spaCy splits French elisions (*l', d', qu'*) and contractions (*du = de + le*, *aux = à + les*).

In [ ]:
doc = nlp("L'énergie d'origine renouvelable qu'on produit aux Pays-Bas réduit l'usage du charbon.")
print([t.text for t in doc])
print([t.lemma_ for t in doc])

## 2. Visualization with displaCy (hand-in item 2)

In [ ]:
doc = nlp("La France a ratifié l'Accord de Paris lors de la COP21 en 2015.")
displacy.render(doc, style='dep', jupyter=True, options={'compact': True, 'distance': 110})

In [ ]:
displacy.render(doc, style='ent', jupyter=True)

In [ ]:
# Save the images as files too (useful for the report / GitHub)
Path('images').mkdir(exist_ok=True)
Path('images/dependency.svg').write_text(
    displacy.render(doc, style='dep', jupyter=False, options={'compact': True, 'distance': 110}), encoding='utf-8')
Path('images/entities.html').write_text(
    displacy.render(doc, style='ent', jupyter=False, page=True), encoding='utf-8')
print('Saved images/dependency.svg and images/entities.html')

## 3. Loading and measuring the corpus (section 6.4)
The texts were collected with `collect_corpus.py` (French Wikipedia API) and saved as UTF-8 files `corpus/text01.txt`, `corpus/text02.txt`, …

In [ ]:
files = sorted(Path('corpus').glob('*.txt'))
texts = [f.read_text(encoding='utf-8') for f in files]
with open('corpus/metadata.csv', encoding='utf-8') as f:
    meta = {row['file']: row for row in csv.DictReader(f)}

docs = list(nlp.pipe(texts))
tokens = [t.text.lower() for d in docs for t in d if t.is_alpha]

print('Texts:', len(texts))
print('Tokens:', len(tokens))
print('Unique words:', len(set(tokens)))
print(Counter(tokens).most_common(10))

**Observation:** as the tutorial predicts, the ten most frequent words are mostly French stop words (*de, la, le, les, des, et, en, …*). They say almost nothing about the topic. Removing stop words and using lemmas shows the actual environmental vocabulary:

In [ ]:
content = [t.lemma_.lower() for d in docs for t in d if t.is_alpha and not t.is_stop]
print('Top 15 content lemmas:', Counter(content).most_common(15))

In [ ]:
# Texts and tokens per category (the labels we will use for classification later)
per_cat = Counter()
tok_cat = Counter()
for f, d in zip(files, docs):
    cat = meta[f.name]['category']
    per_cat[cat] += 1
    tok_cat[cat] += sum(t.is_alpha for t in d)
for cat in per_cat:
    print(f"{cat:<25} texts={per_cat[cat]:<4} tokens={tok_cat[cat]}")

## 4. Looking for quality problems (section 6.5)

In [ ]:
problems = []
for f, text in zip(files, texts):
    if re.search(r'<[^>]+>', text):
        problems.append((f.name, 'HTML tag'))
    if re.search(r'Ã.|Â|â€', text):
        problems.append((f.name, 'mojibake (wrong encoding)'))
    lines = [l for l in text.splitlines() if l.strip()]
    if len(lines) != len(set(lines)):
        problems.append((f.name, 'repeated line'))
    if re.search(r'\(\s*\)|\[\d+\]|Modifier|Article détaillé', text):
        problems.append((f.name, 'Wikipedia leftover (empty brackets / references / edit links)'))
    n = len(text.split())
    if not 100 <= n <= 300:
        problems.append((f.name, f'length {n} words (outside 100–300)'))

# Duplicate texts
seen = {}
for f, text in zip(files, texts):
    if text in seen:
        problems.append((f.name, f'duplicate of {seen[text]}'))
    seen.setdefault(text, f.name)

# Sentences in another language: very few French stop words is suspicious
FR = {'le','la','les','de','des','du','et','est','en','un','une','que','qui','dans','pour','par','sur'}
for f, d in zip(files, docs):
    for s in d.sents:
        words = [t.text.lower() for t in s if t.is_alpha]
        if len(words) >= 8 and sum(w in FR for w in words) / len(words) < 0.05:
            problems.append((f.name, f'possibly non-French sentence: {s.text[:60]}…'))

# Accents check: special characters display correctly
accents = Counter(ch for t in texts for ch in t if ch in 'éèêàâçùûôîïëœÉÈÀÇ')
print('Accented characters found:', dict(accents.most_common()))
print(f'\n{len(problems)} problem(s) found:')
for p in problems:
    print(' -', *p)

### Three sentences from the corpus: one thing right, one thing wrong (hand-in item 3)

In [ ]:
# One sentence from three different categories
chosen = []
for cat in ['politique_climatique', 'energies_renouvelables', 'pollution']:
    f = next(f for f in files if meta[f.name]['category'] == cat)
    d = docs[files.index(f)]
    chosen.append((f.name, next(d.sents).text))

for i, (name, s) in enumerate(chosen, 1):
    print(f'Sentence {i} ({name}): {s}\n')

In [ ]:
for name, s in chosen:
    d = nlp(s)
    displacy.render(d, style='ent', jupyter=True)
    displacy.render(d, style='dep', jupyter=True, options={'compact': True, 'distance': 90})
    print('Entities:', [(e.text, e.label_) for e in d.ents])
    print('POS:', [(t.text, t.pos_) for t in d][:25])
    print('-' * 80)

| # | Sentence | ✔ What the pipeline got right | ✘ What the pipeline got wrong |
|---|---|---|---|
| 1 | *(see output above)* | TODO | TODO |
| 2 | *(see output above)* | TODO | TODO |
| 3 | *(see output above)* | TODO | TODO |

## 5. Comparing the groups — our row (section 6.6)

In [ ]:
row = {
    'Language': 'French',
    'Source': 'Wikipédia (fr), article introductions',
    'Licence': 'CC BY-SA 4.0',
    'Texts': len(texts),
    'Tokens': len(tokens),
    'Unique words': len(set(tokens)),
    'Problems found': len(problems),
}
for k, v in row.items():
    print(f'{k:<15}{v}')

**Notes for the discussion**
- **Accents:** all accented characters are preserved because every file is read and written in UTF-8.
- **Elision / contractions:** see section 1. Elided forms (*l', d', qu'*) are split correctly and lemmatized (*l'* → *le*, *qu'* → *que*). Contractions are **not** split: *du* stays one token with lemma *de*, and *aux* one token with lemma *à*, so the article (*le / les*) is lost.
- **Compounds:** hyphenated names are split, e.g. *Pays-Bas* → `Pays`, `-`, `Bas`.
- **Acronyms:** *ADEME* is tagged as a common noun (NOUN) with lemma *ademe*, not as an organisation. Domain acronyms (ADEME, GIEC, COP21) are hard for the general French model.
- **Before a real project:** add more texts (target ~100, about 25 per category), balance the categories, and consider adding official sources (ADEME, vie-publique.fr, EU) after checking their licences.

## 6. Sources and licences (hand-in item 5)

In [ ]:
print(f"{'File':<12}{'Category':<24}{'Title':<55}Licence")
for f in files:
    m = meta[f.name]
    print(f"{f.name:<12}{m['category']:<24}{m['title'][:53]:<55}{m['licence']}")
print('\nAll texts: Wikipédia (fr), CC BY-SA 4.0 — https://creativecommons.org/licenses/by-sa/4.0/')
print('Full URLs and dates: corpus/metadata.csv')